# 02a — Clinical data cleaning and cohort definition

**Lymphoma AI Research** · Step 2a (clinical only; expression matrices are handled in 02b)

Inputs: raw GEO clinical tables saved in step 1 (`data/raw/*_clinical_raw.csv`).
Outputs: cleaned, harmonised clinical tables, a patient-flow table, a locked train/test split for GSE181063 and a SHA-256 manifest.

**No survival comparison between groups is done in this notebook.** Every filter below uses only eligibility, QC or data-quality information, never outcome-by-group results.

TR: Bu defter sadece klinik verileri temizler, hasta akışını raporlar ve eğitim/test bölmesini kilitler. Gruplar arası sağkalım karşılaştırması yok.

## 1. Setup
TR: Drive'ı bağlar ve klasör yollarını tanımlar. Klasör adın farklıysa sadece `PROJECT_DIR` satırını değiştir.

In [ ]:
import os, sys, glob, json, hashlib
import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# >>> Edit this line if your project folder has a different name <<<
PROJECT_DIR = os.environ.get('LYMPHOMA_PROJECT_DIR', '/content/drive/MyDrive/dlbcl-prognosis')

RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
CLEAN_DIR = os.path.join(PROJECT_DIR, 'data', 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

RAW_FILES = {
    'GSE181063': 'GSE181063_clinical_raw.csv',
    'GSE10846':  'GSE10846_clinical_raw.csv',
    'GSE87371':  'GSE87371_clinical_raw.csv',
}

missing = [f for f in RAW_FILES.values() if not os.path.exists(os.path.join(RAW_DIR, f))]
if missing:
    print('Not found in', RAW_DIR, ':', missing)
    # Help locate the files if the folder name is different
    hits = glob.glob('/content/drive/MyDrive/**/' + missing[0], recursive=True) if IN_COLAB else []
    print('Possible locations:', [os.path.dirname(os.path.dirname(os.path.dirname(h))) for h in hits])
    raise FileNotFoundError('Set PROJECT_DIR to the folder that contains data/raw/')
print('Raw files found in', RAW_DIR)

SEED = 20261014          # fixed seed, written into the manifest
pd.set_option('display.width', 160)

## 2. Load raw files and record their checksums
TR: Ham dosyaları okur ve SHA-256 parmak izlerini kaydeder. Böylece ileride verinin değişmediğini kanıtlayabiliriz.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

raw = {}
raw_hashes = {}
for gse, fname in RAW_FILES.items():
    p = os.path.join(RAW_DIR, fname)
    raw[gse] = pd.read_csv(p)
    raw_hashes[fname] = sha256(p)
    print(f'{gse:10s} {raw[gse].shape}  sha256={raw_hashes[fname][:12]}...')

## 3. Helpers: patient-flow log and IPI
TR: Her filtreden sonra kalan hasta ve olay sayısını kaydeden bir fonksiyon ve IPI'yı bileşenlerinden hesaplayan fonksiyon.

In [ ]:
flow = []
def log_step(cohort, step, df, event_col=None):
    n = len(df)
    e = int(df[event_col].sum()) if (event_col is not None and event_col in df) else None
    flow.append({'cohort': cohort, 'step': step, 'n': n, 'events': e})
    print(f'{cohort:10s} | {step:60s} | n={n:5d}' + (f' | deaths={e}' if e is not None else ''))

def compute_ipi(age, ecog, stage, ldh_high, extranodal):
    """International Prognostic Index (0-5). Returns NaN if any component is missing."""
    parts = pd.concat([
        (age > 60).where(age.notna()),
        (ecog >= 2).where(ecog.notna()),
        (stage >= 3).where(stage.notna()),
        ldh_high,
        (extranodal >= 2).where(extranodal.notna()),
    ], axis=1).astype(float)
    return parts.sum(axis=1, min_count=5)

def reverse_km_median(time, event):
    """Median follow-up by reverse Kaplan-Meier (censoring treated as the event)."""
    t = np.asarray(time, float); c = 1 - np.asarray(event, int)
    order = np.argsort(t); t, c = t[order], c[order]
    s, n = 1.0, len(t)
    for i, (ti, ci) in enumerate(zip(t, c)):
        if ci:
            s *= 1 - 1 / (n - i)
            if s <= 0.5:
                return ti
    return np.nan

## 4. GSE181063 (discovery cohort, HMRN, Illumina DASL)
Eligibility (decided before any outcome analysis):
1. Diagnosis **DLBCL, NOS** only (removes PMBL, THRLBCL, plasmablastic, CNS and non-DLBCL entities)
2. `qc_fail == 0`
3. First-line `CHOP-R` (bortezomib-arm and other regimens excluded)
4. `curative_intent == 1`
5. Valid OS: follow-up time > 0
6. **One array per patient** (the series contains repeat arrays of the same patient — see below)
7. Pre-treatment biopsy only (`sample_pre_active_treatment == 'Yes'`), switchable flag

TR: Keşif setine eleme ölçütlerini sırayla uygular ve her adımda kalan sayıyı yazar.

In [ ]:
EXCLUDE_NON_PRETREATMENT = True   # pre-specified; set False only as a sensitivity analysis

a = raw['GSE181063'].copy()

# --- survival coding check: os_status 1 = dead, 0 = alive ---
assert set(a['os_status'].dropna().unique()) <= {0, 1}
med = a.groupby('os_status')['os_followup_y'].median()
print('Median follow-up (years) by os_status:', med.round(2).to_dict())
assert med[1] < med[0], 'Unexpected: dead patients should have shorter follow-up'
a['os_event'] = a['os_status'].astype(int)
a['os_time_months'] = a['os_followup_y'] * 12

log_step('GSE181063', 'all samples in series', a, 'os_event')
a = a[a['case_diagnostic_subtype_icdo3'] == 'Diffuse large B-cell lymphoma, NOS']
log_step('GSE181063', 'DLBCL, NOS', a, 'os_event')
a = a[a['qc_fail'] == 0]
log_step('GSE181063', 'qc_fail == 0', a, 'os_event')
a = a[a['firstline_regimen'] == 'CHOP-R']
log_step('GSE181063', 'first-line CHOP-R', a, 'os_event')
a = a[a['curative_intent'] == 1]
log_step('GSE181063', 'curative intent', a, 'os_event')
a = a[a['os_time_months'] > 0]
log_step('GSE181063', 'OS time > 0', a, 'os_event')

### 4b. Duplicate patients in GSE181063
Some patients have 2–4 arrays (re-hybridisations or repeat biopsies). If the same patient lands in both training and test sets, the test set is no longer independent — this is **data leakage**. We group arrays by a clinical fingerprint (age at diagnosis, sex, OS time, OS status) and keep one array per patient, preferring a pre-treatment sample with the best probe detection rate. This choice uses QC metrics only, not outcome.

TR: Aynı hastanın birden fazla array'i var. Eğitim ve teste aynı hasta düşerse test bağımsız olmaz (veri sızıntısı). Her hastadan bir örnek tutuyoruz; seçim sonuca değil kaliteye göre.

In [ ]:
fp_cols = ['age_at_diagnosis', 'Sex', 'os_followup_y', 'os_status']
a['patient_key'] = 'G181_' + a.groupby(fp_cols, dropna=False).ngroup().astype(str)

# cross-check against the patient IDs published for a subset (pid_pmid_32187361)
pid = a.dropna(subset=['pid_pmid_32187361'])
inconsistent = pid.groupby('pid_pmid_32187361')['patient_key'].nunique()
print('Published PIDs mapping to >1 fingerprint:', int((inconsistent > 1).sum()))
fp_to_pid = pid.groupby('patient_key')['pid_pmid_32187361'].nunique()
print('Fingerprints mapping to >1 published PID (possible false merge):', int((fp_to_pid > 1).sum()))

sizes = a['patient_key'].value_counts()
print(f'Arrays: {len(a)} | unique patients: {sizes.size} | patients with >1 array: {(sizes > 1).sum()}')

a['_pre'] = (a['sample_pre_active_treatment'] == 'Yes').astype(int)
a = (a.sort_values(['patient_key', '_pre', 'probe_detection'], ascending=[True, False, False])
       .drop_duplicates('patient_key', keep='first'))
log_step('GSE181063', 'one array per patient', a, 'os_event')

if EXCLUDE_NON_PRETREATMENT:
    a = a[a['_pre'] == 1]
    log_step('GSE181063', 'pre-treatment biopsy', a, 'os_event')

### 4c. Clinical variables: sentinel codes and IPI check
The HMRN table uses codes such as 8 (ECOG), 888/999 (albumin), 88.8 (Hb) and 8888 for "unknown". They are set to missing. We recompute IPI from its components and compare with the provided `ipi_score`.

TR: "Bilinmiyor" anlamına gelen kodları (8, 888, 8888…) eksik değere çevirir ve verilen IPI ile bizim hesapladığımız IPI'yı karşılaştırır.

In [ ]:
a['ecog'] = a['performance_status_ecog'].where(a['performance_status_ecog'].between(0, 4))
stage_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
a['stage'] = a['Stage'].map(stage_map)
a['ldh_high'] = a['ldh'].map({'raised': 1.0, 'normal': 0.0, 'low': 0.0})   # 'not done'/'unknown' -> NaN
a['extranodal'] = pd.to_numeric(a['num_extranodal'], errors='coerce')
a['ipi_recomputed'] = compute_ipi(a['age_at_diagnosis'], a['ecog'], a['stage'], a['ldh_high'], a['extranodal'])

both = a.dropna(subset=['ipi_score', 'ipi_recomputed'])
print(f'IPI provided: {a.ipi_score.notna().sum()}/{len(a)} | recomputable: {a.ipi_recomputed.notna().sum()}/{len(a)}')
print(f'Agreement where both exist: {(both.ipi_score == both.ipi_recomputed).mean():.1%} (n={len(both)})')

clean_181 = pd.DataFrame({
    'sample_id': a['gsm'], 'array_id': a['title'], 'patient_key': a['patient_key'],
    'cohort': 'GSE181063', 'platform': a['platform'],
    'os_time_months': a['os_time_months'], 'os_event': a['os_event'],
    'age': a['age_at_diagnosis'], 'sex': a['Sex'].map({'M': 'M', 'F': 'F'}),
    'stage': a['stage'], 'ecog': a['ecog'], 'ldh_high': a['ldh_high'], 'extranodal': a['extranodal'],
    'ipi': a['ipi_score'].fillna(a['ipi_recomputed']),
    'coo': a['pred_combine'].replace({'UNC': 'UNC'}),  # ABC / GCB / MHG / UNC
    'treatment': a['firstline_regimen'],
})
print(clean_181['coo'].value_counts().to_dict())

## 5. GSE10846 (validation, Lenz et al., Affymetrix U133 Plus 2.0)
1. Remove the 6 non-patient CD19-sorted reference samples (no clinical data)
2. R-CHOP-like regimen only
3. OS time > 0
Time is given in **years** → converted to months. `LDH ratio == 0` is physiologically impossible and is treated as missing.

TR: Lenz kohortunda R-CHOP hastalarını seçer, yılı aya çevirir ve imkânsız LDH değerlerini eksik sayar.

In [ ]:
b = raw['GSE10846'].copy()
b = b.rename(columns=lambda c: c.replace('Clinical info: ', ''))
assert set(b['Follow up status'].dropna().unique()) <= {'DEAD', 'ALIVE'}
b['os_event'] = (b['Follow up status'] == 'DEAD').astype(int)
b['os_time_months'] = b['Follow up years'] * 12

log_step('GSE10846', 'all samples in series', b, 'os_event')
b = b[b['Individual'].notna() & b['Final microarray diagnosis'].notna()]
log_step('GSE10846', 'patient samples (reference arrays removed)', b, 'os_event')
b = b[b['Chemotherapy'] == 'R-CHOP-Like Regimen']
log_step('GSE10846', 'R-CHOP-like', b, 'os_event')
b = b[b['os_time_months'] > 0]
log_step('GSE10846', 'OS time > 0', b, 'os_event')
print('Duplicate patients (Individual):', int(b['Individual'].duplicated().sum()))

b['ldh_ratio'] = b['LDH ratio'].where(b['LDH ratio'] > 0)
b['ldh_high'] = (b['ldh_ratio'] > 1).astype(float).where(b['ldh_ratio'].notna())
b['ipi'] = compute_ipi(b['Age'], b['ECOG performance status'], b['Stage'], b['ldh_high'],
                       b['Number of extranodal sites'])
print(f"LDH missing: {b.ldh_ratio.isna().sum()} | IPI computable: {b.ipi.notna().sum()}/{len(b)}")

clean_108 = pd.DataFrame({
    'sample_id': b['gsm'], 'array_id': b['title'], 'patient_key': 'G108_' + b['Individual'].astype(str),
    'cohort': 'GSE10846', 'platform': b['platform'],
    'os_time_months': b['os_time_months'], 'os_event': b['os_event'],
    'age': b['Age'], 'sex': b['Gender'].map({'male': 'M', 'female': 'F'}),
    'stage': b['Stage'], 'ecog': b['ECOG performance status'], 'ldh_high': b['ldh_high'],
    'extranodal': b['Number of extranodal sites'], 'ipi': b['ipi'],
    'coo': b['Final microarray diagnosis'].map({'ABC DLBCL': 'ABC', 'GCB DLBCL': 'GCB', 'Unclassified DLBCL': 'UNC'}),
    'treatment': b['Chemotherapy'],
})
print(clean_108['coo'].value_counts().to_dict())

## 6. GSE87371 (validation, LYSA, Affymetrix U133 Plus 2.0)
**Critical coding check.** In this series `cens_os = 1` means **censored (alive)** and `cens_os = 0` means **death**. Evidence from the data itself: there is no patient with PFS censored but OS = 0 (a death always counts as progression), and every patient with `cens_pfs = 0` but `cens_os = 1` has PFS time ≤ OS time (progressed, still alive). Using `cens_os` directly as the event would invert every result.

1. Valid survival data
2. R-CHOP family (R-CHOP14, R-CHOP21, R-mini-CHOP21); ACVBP-based arms excluded
3. PMBL excluded
4. OS time > 0

Pre-specified: R-mini-CHOP (dose-reduced, elderly) stays in the primary set and is flagged for a sensitivity analysis.

TR: Bu sette kodlama ters: 0 = ölüm, 1 = sansür. Bunu verinin kendisiyle kanıtlayıp olay değişkenini 1 − cens_os olarak tanımlıyoruz.

In [ ]:
c = raw['GSE87371'].copy()
c = c[c['os_time'].notna() & c['cens_os'].notna()]

# evidence for the coding (asserted, not assumed)
assert ((c['cens_pfs'] == 1) & (c['cens_os'] == 0)).sum() == 0, 'death without progression found: recheck coding'
assert (c['pfs_time'] <= c['os_time'] + 1e-6).all()
print(pd.crosstab(c['cens_pfs'], c['cens_os']))
c['os_event'] = (1 - c['cens_os']).astype(int)
c['os_time_months'] = c['os_time']   # already in months (max ~81)

log_step('GSE87371', 'samples with survival data', c, 'os_event')
c = c[c['treatment'].isin(['R-CHOP14', 'R-CHOP21', 'R-Mini-CHOP21'])]
log_step('GSE87371', 'R-CHOP family', c, 'os_event')
c = c[c['coo'] != 'PMBL']
log_step('GSE87371', 'PMBL excluded', c, 'os_event')
c = c[c['os_time_months'] > 0]
log_step('GSE87371', 'OS time > 0', c, 'os_event')
print('R-mini-CHOP patients (sensitivity flag):', int((c['treatment'] == 'R-Mini-CHOP21').sum()))

clean_873 = pd.DataFrame({
    'sample_id': c['gsm'], 'array_id': c['ID'], 'patient_key': 'G873_' + c['ID'].astype(str),
    'cohort': 'GSE87371', 'platform': c['platform'],
    'os_time_months': c['os_time_months'], 'os_event': c['os_event'],
    'age': c['age'], 'sex': c['Sex'].map({'MALE': 'M', 'FEMALE': 'F'}),
    'stage': c['Stage'].str.extract(r'(\d)')[0].astype(float),
    'ecog': np.nan, 'ldh_high': np.nan, 'extranodal': np.nan,   # components not published
    'ipi': c['ipi'],
    'coo': c['coo'].map({'ABC': 'ABC', 'GC': 'GCB', 'Other': 'UNC'}),
    'treatment': c['treatment'],
})
print(clean_873['coo'].value_counts().to_dict())

## 7. Harmonised summary
TR: Üç kohortu tek formatta birleştirir ve özet tabloyu (hasta, ölüm, medyan takip, IPI eksikliği, COO dağılımı) gösterir.

In [ ]:
clean_all = pd.concat([clean_181, clean_108, clean_873], ignore_index=True)
assert clean_all['sample_id'].is_unique and clean_all['patient_key'].is_unique
assert clean_all['os_event'].isin([0, 1]).all() and (clean_all['os_time_months'] > 0).all()

rows = []
for coh, d in clean_all.groupby('cohort', sort=False):
    rows.append({
        'cohort': coh, 'platform': d['platform'].iloc[0], 'patients': len(d),
        'deaths': int(d['os_event'].sum()),
        'median_follow_up_months': round(reverse_km_median(d['os_time_months'], d['os_event']), 1),
        'median_age': d['age'].median(), 'IPI_missing': int(d['ipi'].isna().sum()),
        **{f'COO_{k}': int(v) for k, v in d['coo'].value_counts().items()},
    })
summary = pd.DataFrame(rows).fillna(0)
summary

## 8. Lock the GSE181063 train/test split (70/30, stratified by OS event)
The split is made **now**, on clinical data only, before any expression data is loaded. The test set is not touched again until step 5. Positive control (step 3) and all model building (step 4) use the training set only.

TR: Eğitim/test bölmesini şimdi, ifade verisine hiç bakmadan yapıp dosyaya kaydediyoruz. Test seti 5. adıma kadar açılmayacak.

In [ ]:
from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(
    clean_181['sample_id'], test_size=0.30, random_state=SEED, stratify=clean_181['os_event'])
split = pd.DataFrame({'sample_id': clean_181['sample_id']})
split['set'] = np.where(split['sample_id'].isin(train_ids), 'train', 'test')

chk = clean_181.merge(split, on='sample_id')
assert chk.groupby('patient_key')['set'].nunique().max() == 1   # no patient in both sets
print(chk.groupby('set').agg(patients=('sample_id', 'size'), deaths=('os_event', 'sum')))
# Only counts are printed. No survival comparison between train and test.

## 9. Save outputs and write the manifest
TR: Temiz dosyaları, hasta akış tablosunu ve bölmeyi kaydeder; tüm dosyaların SHA-256 değerlerini bir manifest dosyasına yazar.

In [ ]:
out = {
    'clinical_clean_GSE181063.csv': clean_181,
    'clinical_clean_GSE10846.csv': clean_108,
    'clinical_clean_GSE87371.csv': clean_873,
    'clinical_clean_all.csv': clean_all,
    'patient_flow.csv': pd.DataFrame(flow),
    'cohort_summary.csv': summary,
    'split_GSE181063.csv': split,
}
for fname, df in out.items():
    df.to_csv(os.path.join(CLEAN_DIR, fname), index=False)

manifest = {
    'notebook': '02a_clinical_cleaning.ipynb',
    'seed': SEED,
    'exclude_non_pretreatment': EXCLUDE_NON_PRETREATMENT,
    'raw_inputs_sha256': raw_hashes,
    'outputs_sha256': {f: sha256(os.path.join(CLEAN_DIR, f)) for f in out},
    'pandas': pd.__version__, 'numpy': np.__version__,
}
with open(os.path.join(CLEAN_DIR, 'manifest_02a.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

print('Saved to', CLEAN_DIR)
for f in list(out) + ['manifest_02a.json']:
    print('  ', f)
print('\nDone. Copy the patient-flow and summary tables above into the chat, then end this Colab session (Runtime > Disconnect and delete runtime).')